# Run the real pipeline

Unlike `build_demo_db.ipynb` (hand-authored, illustrative data, no network
calls), this notebook calls `protein_selector.pipeline.run_pipeline` for
real: real RCSB hard-filters search, real simulability/composition checks,
real ligand CCD/SMILES lookup, real RDKit (and Meeko, if installed)
parameterizability, real Europe PMC literature counts, and a real AlphaFold
DB lookup for ex02 -- every number below comes from an actual API response,
nothing is invented.

**Stage config is grouped into dataclasses** (`CandidateSearchConfig`, `CandidateFilterConfig`, `MdSimulationConfig`,
`PocketDetectionConfig`, ...) -- see `pipeline.py`'s own docstring for the full
list and what each corresponds to (`ex02`/`ex03`/`ex04` are still the internal
persisted-column labels, but the pipeline's own API and config classes describe
what each stage actually does instead of that jargon).

MD simulation and pocket detection need the conda-only
`environment-validation.yml` env (a local `fpocket` binary too, for pocket
detection) -- this notebook enables both below, so it must be run through that
conda env's Python (the "Python 3 (protein-selector-validation, conda)" kernel),
not the base venv. Docking (`ex04`, real Vina + PLIP) still isn't wired into the
pipeline at all -- no receptor-prep/pocket-center-extraction code exists yet to
auto-run it, so it will always show `"not_run"` regardless of config.

`max_candidates` is kept small (a real Meeko 3D-embedding step, if
installed, can be slow for some real bound ligands) -- raise it once you've
confirmed a run completes in reasonable time for your machine.

Requires the `notebook` dependency group: `uv sync --group notebook`.

In [1]:
import logging
from pathlib import Path

import pandas as pd

from protein_selector.pipeline import (
    CandidateFilterConfig,
    CandidateSearchConfig,
    MdSimulationConfig,
    PocketDetectionConfig,
    run_pipeline,
)

logging.basicConfig(level=logging.INFO, format="%(message)s")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Separate db from build_demo_db.ipynb's illustrative one, so the two never mix.
DB_PATH = Path("cache/protein_selector_real.db")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
CSV_PATH = Path("report_real.csv")

/home/yescalona/.local/share/micromamba/envs/protein-selector-validation/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
rows = run_pipeline(
    db_path=DB_PATH,
    candidate_search=CandidateSearchConfig(max_candidates=1000, random_seed=42),
    # candidates bigger than 50 residues are dropped from the pool entirely --
    # see CandidateFilterConfig's own docstring for why residues, not atoms.
    candidate_filter=CandidateFilterConfig(max_residues=100),
    # modeling_lookup left at its default (ModelingLookupConfig(enabled=True))
    # -- the real AlphaFold DB fetch-only check, persisted as exercise "ex02".
    md_simulation=MdSimulationConfig(
        enabled=True,
        n_steps=50,
        max_minimization_iterations=10,
    ),
    pocket_detection=PocketDetectionConfig(enabled=True),
    report_csv_path=CSV_PATH,
)
len(rows)

🔎 search: pool_size 20000 exceeds RCSB's 10000-row ceiling, clamping
HTTP Request: POST https://search.rcsb.org/rcsbsearch/v2/query "HTTP/1.1 200 OK"
HTTP Request: POST https://search.rcsb.org/rcsbsearch/v2/query "HTTP/1.1 200 OK"
🔎 search: sampled 1000 of 10000 matching candidates
HTTP Request: POST https://data.rcsb.org/graphql "HTTP/1.1 200 OK"
HTTP Request: POST https://data.rcsb.org/graphql "HTTP/1.1 200 OK"
HTTP Request: POST https://data.rcsb.org/graphql "HTTP/1.1 200 OK"
HTTP Request: POST https://data.rcsb.org/graphql "HTTP/1.1 200 OK"
HTTP Request: POST https://data.rcsb.org/graphql "HTTP/1.1 200 OK"
HTTP Request: POST https://data.rcsb.org/graphql "HTTP/1.1 200 OK"
HTTP Request: POST https://data.rcsb.org/graphql "HTTP/1.1 200 OK"
HTTP Request: POST https://data.rcsb.org/graphql "HTTP/1.1 200 OK"
HTTP Request: POST https://data.rcsb.org/graphql "HTTP/1.1 200 OK"
HTTP Request: POST https://data.rcsb.org/graphql "HTTP/1.1 200 OK"
HTTP Request: POST https://data.rcsb.org/graphq

135

## The real joined report

In [3]:
from protein_selector.core.report import rows_to_dataframe

report_df = rows_to_dataframe(rows)
report_df

,pdb_id,uniprot_id,title,organism,n_residues,n_atoms,resolution,method,n_protein_entities,ligand_ccd,ligand_smiles,ligand_parameterizable,ligand_meeko_parameterizable,pocket_found,pocket_score,completeness,nonstd_residues,litref_count,suitable_for,ex02_status,ex02_predicted_difficulty,ex02_measured_difficulty,ex02_gap,ex02_tier,ex02_failure_mode,ex02_notes,ex03_status,ex03_predicted_difficulty,ex03_measured_difficulty,ex03_gap,ex03_tier,ex03_failure_mode,ex03_notes,ex04_status,ex04_predicted_difficulty,ex04_measured_difficulty,ex04_gap,ex04_tier,ex04_failure_mode,ex04_notes,rationale_json
0,101M,P02185,SPERM WHALE MYOGLOBIN F46V N-BUTYL ISOCYANIDE ...,Physeter macrocephalus,154,1413,2.07,X-RAY DIFFRACTION,1,HEM,None,True,False,None,NaN,1.000000,False,0.0,"[""ex02""]",pass,0.000,0.031135,0.031135,intro,NaN,"[AF-P02185-F1: mean pLDDT 97.5, 0.0% low/very-...",not_run,0.447111,NaN,NaN,core,NaN,[],not_run,1.0000,None,None,challenge,None,[],"{""simulability_reasons"": [], ""pocket_reasons"":..."
1,102L,P00720,HOW AMINO-ACID INSERTIONS ARE ALLOWED IN AN AL...,Tequatrovirus T4,165,1439,1.74,X-RAY DIFFRACTION,1,BME,None,True,True,None,NaN,0.987879,False,5.0,[],fail,NaN,1.000000,NaN,challenge,completeness,[no AlphaFold DB entry found for UniProt acces...,not_run,0.419374,NaN,NaN,core,NaN,[],not_run,0.0000,None,None,intro,None,[],"{""simulability_reasons"": [], ""pocket_reasons"":..."
2,102M,P02185,SPERM WHALE MYOGLOBIN H64A AQUOMET AT PH 9.0,Physeter macrocephalus,154,1423,1.84,X-RAY DIFFRACTION,1,HEM,None,True,False,None,NaN,1.000000,False,0.0,"[""ex02""]",pass,0.000,0.032788,0.032788,intro,NaN,"[AF-P02185-F1: mean pLDDT 97.5, 0.0% low/very-...",not_run,0.416444,NaN,NaN,core,NaN,[],not_run,1.0000,None,None,challenge,None,[],"{""simulability_reasons"": [], ""pocket_reasons"":..."
3,10PA,Q2G0L4,Crystal structure of SdrD A2-A3 domains from S...,Staphylococcus aureus subsp. aureus JH1,321,2574,1.92,X-RAY DIFFRACTION,1,CA,None,True,True,None,NaN,0.987539,True,0.0,[],fail,0.358,1.000000,0.642000,challenge,confidence,"[35.8% of AF-Q2G0L4-F1 is low/very-low pLDDT, ...",not_run,0.593487,NaN,NaN,core,NaN,[],not_run,0.0000,None,None,intro,None,[],"{""simulability_reasons"": [""residue count 321 o..."
4,107M,P02185,SPERM WHALE MYOGLOBIN V68F N-BUTYL ISOCYANIDE ...,Physeter macrocephalus,154,1418,2.09,X-RAY DIFFRACTION,1,HEM,None,True,False,None,NaN,1.000000,True,0.0,"[""ex02""]",pass,0.000,0.038397,0.038397,intro,NaN,"[AF-P02185-F1: mean pLDDT 97.5, 0.0% low/very-...",not_run,0.449778,NaN,NaN,core,NaN,[],not_run,1.0000,None,None,challenge,None,[],"{""simulability_reasons"": [""residue count 154 o..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130,1C53,P00120,S-CLASS CYTOCHROMES C HAVE A VARIETY OF FOLDIN...,Nitratidesulfovibrio vulgaris str. 'Miyazaki F,79,122,1.80,X-RAY DIFFRACTION,1,HEM,None,True,False,True,0.648,1.000000,False,0.0,"[""ex02"", ""ex03""]",pass,0.216,0.037553,-0.178447,intro,NaN,"[AF-P00120-F1: mean pLDDT 86.9, 21.6% low/very...",pass,0.327778,0.013358,-0.314420,intro,NaN,"[1168 atoms, 50 steps at 2.0 fs completed clea...",not_run,0.6760,None,None,challenge,None,[],"{""simulability_reasons"": [], ""pocket_reasons"":..."
131,1DXG,P00273,CRYSTAL STRUCTURE OF DESULFOREDOXIN FROM DESUL...,Megalodesulfovibrio gigas,72,593,1.80,X-RAY DIFFRACTION,1,FE,None,True,True,False,0.031,1.000000,False,5.0,"[""ex02"", ""ex03""]",pass,0.000,0.037367,0.037367,intro,NaN,"[AF-P00273-F1: mean pLDDT 96.6, 0.0% low/very-...",pass,0.320000,0.006929,-0.313071,intro,NaN,"[1034 atoms, 50 steps at 2.0 fs completed clea...",not_run,0.4845,None,None,core,None,[],"{""simulability_reasons"": [], ""pocket_reasons"":..."
132,1CTF,P0A7K2,STRUCTURE OF THE C-TERMINAL DOMAIN OF THE RIBO...,Escherichia coli,74,554,1.70,X-RAY DIFFRACTION,1,SO4,None,True,True,False,0.302,0.918919,False,99.0,"[""ex02"", ""ex03""]",pass,0.231,0.039904,-0.191096,intro,

## What's real here, spelled out

- `title`/`organism`/`n_residues`/`resolution`/... -- real RCSB entry metadata.
- `ligand_parameterizable` -- a real RDKit sanitization result (and real Meeko
  3D-embed + PDBQT-write result, if the `validate` extra is installed).
- `litref_count` -- a real Europe PMC hit count for this exact PDB ID.
- `ex02_status`/`ex02_predicted_difficulty` -- a real AlphaFold DB lookup: if
  the candidate's UniProt accession has a modeled entry, this reflects its
  actual published confidence fractions; if not, `ex02_status` is `"fail"`
  with `FailureMode.COMPLETENESS`, not guessed.

In [4]:
report_df[["pdb_id", "uniprot_id", "ligand_ccd", "ligand_parameterizable", "litref_count", "ex02_status", "ex02_predicted_difficulty", "ex02_failure_mode"]]

,pdb_id,uniprot_id,ligand_ccd,ligand_parameterizable,litref_count,ex02_status,ex02_predicted_difficulty,ex02_failure_mode
0,101M,P02185,HEM,True,0.0,pass,0.000,NaN
1,102L,P00720,BME,True,5.0,fail,NaN,completeness
2,102M,P02185,HEM,True,0.0,pass,0.000,NaN
3,10PA,Q2G0L4,CA,True,0.0,fail,0.358,confidence
4,107M,P02185,HEM,True,0.0,pass,0.000,NaN
...,...,...,...,...,...,...,...,...
130,1C53,P00120,HEM,True,0.0,pass,0.216,NaN
131,1DXG,P00273,FE,True,5.0,pass,0.000,NaN
132,1CTF,P0A7K2,SO4,True,99.0,pass,0.231,NaN
133,1B2D,P01315,SO4,True,4.0,fail,0.861,confidence
